Adaptive Market-Making — Findings & Implementation Notes

## Overview

This session built the market-making simulator and adaptive strategy on top of the
toxicity classifier from Sessions 8-9. The goal was to test whether the toxicity signal
produces measurable improvement in market-making PnL across three assets and three
regime weeks.

---

## Finding 1 — The Infinite-Horizon A-S Spread Collapses to One Tick on BTC Perps

The classical Avellaneda-Stoikov (2008) spread formula in the infinite-horizon limit is:

    δ* = (2/γ) × ln(1 + γ/κ)

At high trade frequency (κ = 19.13 trades/bar on BTC), γ/κ is small and the Taylor
expansion ln(1 + x) ≈ x gives:

    δ* ≈ (2/γ) × (γ/κ) = 2/κ ≈ $0.105

This equals one minimum tick ($0.10) and is completely independent of γ. The risk
aversion parameter cannot control inventory through spread adjustment on this venue.

**Implication:** The A-S spread formula is inoperative on BTC perps. Inventory control
must be implemented through reservation price skew instead.

---

## Finding 2 — κ is Not Portable Across Assets

κ in the original A-S paper has units of 1/dollar — it parameterises fill probability
as a function of quote distance in price space. Estimating κ as raw trades per bar
produces a dimensionless count that happens to align with the dollar spread on BTC
(κ=19, spread=$0.10, 2/κ=$0.105) but breaks completely on ETH and SOL where the spread
is $0.01. At κ=12, the A-S spread formula gives $0.17 on ETH — 17x wider than the
actual market spread, producing zero fills.

**Resolution:** Use the prevailing market spread directly as the MM spread. This is
consistent with the infinite-horizon A-S result and is standard practice in crypto MM
implementations (confirmed in Hummingbot documentation and HangukQuant research blog).

---

## Finding 3 — Fill Size Must Be Volatility-Adjusted and Notional-Consistent

Using average aggressor trade size as fill size is incorrect — it makes the MM absorb
the full one-sided volume of each bar, which blows up inventory immediately on ETH/SOL
and produces artificial PnL from extreme fill prices.

The correct approach derives fill size from a risk budget:

    fill_size = α × spread / (σ × mid)

Where α is the adverse selection tolerance (fraction of a spread the MM accepts losing
per fill on average). This makes fill size:
- Smaller when volatility is high (each fill carries more inventory risk)
- Larger when spread is wide (more revenue per fill justifies larger position)
- Consistent in notional terms across assets

With α=0.5: BTC fill_size ≈ 0.011 BTC ($668), ETH ≈ 0.020 ETH ($47), SOL ≈ 0.32 SOL ($43).

The ETH/SOL notional is small relative to BTC — reflecting that ETH and SOL have worse
spread-to-volatility ratios, making them genuinely harder venues for passive market making.
This is a finding, not a calibration failure.

---

## Finding 4 — The Baseline MM Loses Money, As Expected

A passive market maker in any market loses to adverse selection on average. The spread
earned per fill ($0.05 half-spread × 0.011 BTC = $0.0006) is small relative to the
expected adverse price move over the next 10 bars (σ√10 × mid × fill_size ≈ $0.025).
The ratio is approximately 40:1 against the MM.

This is not a bug. It is the adverse selection problem — the same problem the toxicity
classifier was built to address. The baseline MM's negative Sharpe is the benchmark
against which the adaptive strategy is measured.

Baseline MtM across assets and regimes (k=0, static spread):

| Asset | Week 1 | Week 2 | Week 3 |
|-------|--------|--------|--------|
| BTC   | -$1,937 | -$6,110 | -$989 |
| ETH   | -$406  | -$478  | -$156 |
| SOL   | -$370  | -$463  | -$261 |

---

## Finding 5 — Adaptive Spread Widening Improves Out-of-Sample PnL on BTC and SOL

The adaptive strategy widens the spread proportionally to the per-bar toxicity rate:

    adaptive_spread = market_spread × (1 + k × toxic_rate)

Key design constraint: fill_size and max_inventory are computed from market_spread only,
not adaptive_spread. Using adaptive_spread for sizing caused the MM to take larger
positions on toxic bars — the opposite of the intended effect.

k was calibrated on week 1 only (in-sample). k=5 was selected as the best value
based on week 1 BTC MtM. Applied unchanged to weeks 2 and 3 (out-of-sample).

**Out-of-sample results (week 2 + week 3, k=5 vs k=0):**

| Asset | Baseline (k=0) | Adaptive (k=5) | Improvement |
|-------|---------------|----------------|-------------|
| BTC week2 | -$6,110 | -$5,567 | +$543 |
| BTC week3 | -$989  | +$642  | +$1,631 |
| ETH week2 | -$478  | -$548  | -$70 |
| ETH week3 | -$156  | -$92   | +$64 |
| SOL week2 | -$463  | -$405  | +$58 |
| SOL week3 | -$261  | -$57   | +$204 |

**The adaptive strategy consistently improves BTC and SOL performance out-of-sample.**
The largest improvements are in week 3 (stress regime) — BTC +$1,631 and SOL +$204 —
where toxic rates are highest (14.1% and 23.4% respectively). This is directionally
consistent with the hypothesis: a toxicity signal is most valuable when informed flow
is most concentrated.

ETH week 2 is the one clear failure (-$70). ETH has the weakest toxicity signal
(lowest SHAP contributions, most regime-dependent features) and its week 2 is a
directional breakout regime where the toxicity label is less reliable.

---

## Implementation Decisions

**Quote formula:** Reservation price skew with market spread.
```
skew              = -q × γ × σ × S
reservation_price = mid + skew
mm_bid            = reservation_price - half_spread
mm_ask            = reservation_price + half_spread
```

**Fill condition (Option A):** MM fills if aggressor crossed the quoted price.
```
Sell aggressor: mm_bid >= min_sell_price
Buy  aggressor: mm_ask <= max_buy_price
```

**Fill price:** MM's quoted price (mm_bid or mm_ask), not mid or aggressor extreme price.
Using mid understates spread revenue. Using aggressor extreme price (min_sell_price,
max_buy_price) gave artificial profits of $40k+/week by crediting the MM with the
best possible price in each bar.

**Cash accounting:**
```
MM buys  (sell aggressor): cash -= mm_bid × fill_size, inventory += fill_size
MM sells (buy aggressor):  cash += mm_ask × fill_size, inventory -= fill_size
MtM PnL = cash + inventory × mid
```

**Dynamic parameters per bar:**
```python
fill_size     = alpha * market_spread / (sigma * mid)   # alpha = 0.5
max_inventory = N * fill_size                           # N = 20
gamma         = (market_spread/2) / (M * fill_size * sigma * mid)  # M = 10
```

---

## Limitations

**1. Bar-level toxicity signal vs trade-level execution.** The adaptive strategy adjusts
quotes using the toxic_rate from the previous bar — a 1-second lag. In a real system,
this signal would need to be sub-millisecond. The bar-level implementation understates
the achievable improvement in a production system.

**2. k is calibrated on one asset.** k=5 was chosen based on BTC week 1 only. A more
rigorous approach would calibrate per-asset or use a held-out period from each asset.

**3. Sharpe is unreliable at weekly frequency.** With 6-7 daily observations per week,
annualised Sharpe has standard error of ~1/√6 × √365 ≈ 7.8. MtM improvement is the
more reliable metric at this data volume.

**4. Cartea & Sánchez-Betancourt (2025) analytical comparison not yet implemented.**
Their closed-form price adjustment provides a theory-derived benchmark that does not
require classifier training. This is the next session's deliverable.

---

## Next Session

- Implement Cartea & Sánchez-Betancourt (2025) analytical adjustment
- Three-way comparison: static baseline vs adaptive ML vs analytical
- Phase 5: README, code cleanup, GitHub

Kappa in A-S has units of 1/dollar and must be estimated from fill probability curves as a function of quote distance. Without full LOB depth data, this estimation is not possible. We therefore set the MM spread equal to the prevailing best bid-ask spread, consistent with the infinite-horizon result that the optimal spread converges to a market-determined constant at high κ. Inventory control is implemented entirely through reservation price skew, which is the economically meaningful component of A-S on tick-constrained crypto venues.

In [32]:
import numpy as np
import pandas as pd

def compute_as_quotes(
    best_bid: float,
    best_ask: float,
    mid: float,
    sigma: float,       # rolling vol, in return space (e.g. 0.000079)
    inventory: float,   # current inventory in base asset (e.g. BTC)
    gamma: float,       # risk aversion parameter
    adaptive_spread: float
) -> tuple[float, float, float, float]:
    """
    Infinite-horizon Avellaneda-Stoikov quotes with inventory skew.

    Returns: (mm_bid, mm_ask, spread, skew)
    """

    # Step 1: Infinite-horizon A-S spread
    # Formula: (2/gamma) * ln(1 + gamma/kappa)
    as_spread = adaptive_spread

    # Step 2: Inventory skew
    # Formula: -q * gamma * sigma * S
    skew = -inventory * gamma * sigma * mid

    # Step 3: Reservation price (mid shifted by skew)
    reservation_price = mid + skew

    # Step 4: Place quotes symmetrically around reservation price
    # bid = reservation_price - half_spread
    # ask = reservation_price + half_spread
    half_spread = as_spread / 2
    mm_bid = reservation_price - half_spread
    mm_ask = reservation_price + half_spread

    return mm_bid, mm_ask, as_spread, skew

  
#unit test
# print("no inventory")
# print(compute_as_quotes(59999.95, 60000.05, 60000, 0.000079, 19, 0, 0.1))

# print("with inventory")
# print(compute_as_quotes(59999.95, 60000.05, 60000, 0.000079, 19, 2, 0.1))

In [33]:
def run_backtest(
    bars: pd.DataFrame,
    k: int,
    alpha: float = 0.5,
    N: float = 20,
    M: float = 10,
    sigma_window: int = 300,
) -> pd.DataFrame:

    cash = 0.0
    inventory = 0.0
    records = []

    log_ret = np.log(bars['mid']).diff().fillna(0).values
    sigma_arr = pd.Series(log_ret).rolling(sigma_window, min_periods=1).std().values
    k_values = [0, 1, 2, 5, 10, 20, 50]
    for i, row in bars.iterrows():
        mid      = row['mid']
        best_bid = row['best_bid']
        best_ask = row['best_ask']
        
        market_spread = best_ask - best_bid
        adaptive_spread = (best_ask - best_bid) * (1 + k * row['toxic_rate'])
        sigma    = sigma_arr[i] if sigma_arr[i] > 0 else 1e-6

        # Dynamic parameters
        fill_size     = alpha * market_spread / (sigma * mid)
        max_inventory = N * fill_size
        gamma         = (market_spread / 2) / (M * fill_size * sigma * mid)

        mm_bid, mm_ask, as_spread, skew = compute_as_quotes(
            best_bid, best_ask, mid, sigma, inventory, gamma, adaptive_spread
        )

        if row['any_sell'] and mm_bid >= row['min_sell_price'] and abs(inventory) < max_inventory:
            cash -= mm_bid * fill_size
            inventory += fill_size

        if row['any_buy'] and mm_ask <= row['max_buy_price'] and abs(inventory) < max_inventory:
            cash += mm_bid * fill_size
            inventory -= fill_size

        mtm = cash + inventory * mid
        records.append({
            'mid': mid, 'mm_bid': mm_bid, 'mm_ask': mm_ask,
            'skew': skew, 'inventory': inventory, 'cash': cash,
            'mtm_pnl': mtm, 'sigma': sigma, 'fill_size': fill_size,
            'max_inventory': max_inventory, 'gamma': gamma,
            'timestamp': row['timestamp'],
        })

    return pd.DataFrame(records)

In [34]:
def build_bars(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate tick-level trades into 1-second bars.
    """
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
    df = df.set_index('datetime')
    
    buys = df[df['sign'] == 1]
    sells = df[df['sign'] == -1]
    
    bars = pd.DataFrame({
        'mid'           : df['mid'].resample('1s').last(),
        'best_bid'      : df['best_bid'].resample('1s').last(),
        'best_ask'      : df['best_ask'].resample('1s').last(),
        'bar_trades'    : df['qty'].resample('1s').count(),
        'any_buy'       : buys['qty'].resample('1s').count() > 0,
        'any_sell'      : sells['qty'].resample('1s').count() > 0,
        'max_buy_price' : buys['price'].resample('1s').max(),
        'min_sell_price': sells['price'].resample('1s').min(),
        'buy_volume'    : buys['qty'].resample('1s').sum(),
        'sell_volume'   : sells['qty'].resample('1s').sum(),
        'toxic_rate'    : df['toxic'].resample('1s').mean(),
        'any_toxic' : df['toxic'].astype(int).resample('1s').max().astype(bool),
        'buy_trades'    : buys['qty'].resample('1s').count(),
        'sell_trades'   : sells['qty'].resample('1s').count(),
    })
    
    # Bars with no trades on one side get NaN — fill correctly
    bars['any_buy']  = bars['any_buy'].fillna(False)
    bars['any_sell'] = bars['any_sell'].fillna(False)
    bars['max_buy_price']  = bars['max_buy_price'].fillna(0)
    bars['min_sell_price'] = bars['min_sell_price'].fillna(0)
    bars['buy_volume']  = bars['buy_volume'].fillna(0)
    bars['sell_volume'] = bars['sell_volume'].fillna(0)
    bars['toxic_rate']  = bars['toxic_rate'].fillna(0)
    bars['buy_trades']  = bars['buy_trades'].fillna(0)
    bars['sell_trades'] = bars['sell_trades'].fillna(0)

    # Drop bars with no mid (empty bars)
    bars = bars.dropna(subset=['mid']).reset_index().rename(columns={'datetime': 'timestamp'})
    
    return bars

In [35]:
def summary_stats(results: pd.DataFrame) -> dict:
    # Resample mtm_pnl to daily, compute daily returns
    results.index = pd.to_datetime(results['timestamp'])
    daily_pnl = results['mtm_pnl'].resample('1D').last().diff().dropna()

    mean = daily_pnl.mean()
    std  = daily_pnl.std()
    sharpe = (mean / std * np.sqrt(365)) if std > 0 else 0
    
    

    cummax = results['mtm_pnl'].cummax()
    max_dd = (results['mtm_pnl'] - cummax).min()

    inv_95 = np.percentile(results['inventory'].abs(), 95)
    fill_rate = (results['inventory'].diff().abs() > 0).mean()

    return {
        'sharpe'          : round(sharpe, 3),
        'max_drawdown'    : round(max_dd, 4),
        'final_mtm'       : round(results['mtm_pnl'].iloc[-1], 4),
        'inventory_95pct' : round(inv_95, 4),
        'fill_rate'       : round(fill_rate, 4),
        'mean_bar_pnl'    : round(mean, 6),
        'std_bar_pnl'     : round(std, 6),
    }

In [36]:
#Setup & Data Loading
from pathlib import Path
import pandas as pd

data_dir = Path("../data/processed/features")

BACKTEST_COLS = [
    'timestamp', 'price', 'sign', 'qty',
    'best_bid', 'best_ask', 'midprice', 'toxic'
]

ASSETS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
WEEKS = ['week1', 'week2', 'week3']

BACKTEST_COLS = [
    'timestamp', 'price', 'sign', 'qty',
    'spread', 'midprice', 'toxic'
]

def load_backtest_data(data_dir, asset, week):
    path = Path(data_dir) / f"{asset}_{week}_full_features.parquet"
    df = pd.read_parquet(path, columns=BACKTEST_COLS)
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = df.dropna()
    df['timestamp'] = df['timestamp'].astype('int64') / 1e9
    
    # Reconstruct best bid/ask
    df['best_bid'] = df['midprice'] - df['spread'] / 2
    df['best_ask'] = df['midprice'] + df['spread'] / 2
    df['mid'] = df['midprice']  # alias for backtest engine
    
    return df

# Load as dict, not concatenated DataFrame
all_data = {}
for asset in ASSETS:
    for week in WEEKS:
        all_data[(asset, week)] = load_backtest_data(data_dir, asset, week)
        print(f"  {asset} {week}: {len(all_data[(asset, week)]):,} trades")

  BTCUSDT week1: 10,071,946 trades
  BTCUSDT week2: 12,032,260 trades
  BTCUSDT week3: 24,249,845 trades
  ETHUSDT week1: 4,554,411 trades
  ETHUSDT week2: 7,137,925 trades
  ETHUSDT week3: 10,051,489 trades
  SOLUSDT week1: 3,968,322 trades
  SOLUSDT week2: 5,757,031 trades
  SOLUSDT week3: 11,434,844 trades


In [37]:
# Build bars for all assets/weeks
all_bars = {}
for asset in ASSETS:
    for week in WEEKS:
        bars = build_bars(all_data[(asset, week)])
        all_bars[(asset, week)] = bars
        print(f"{asset} {week}: {len(all_data[(asset, week)]):,} ticks → {len(bars):,} bars "
              f"| avg {all_data[(asset, week)]['qty'].sum()/len(bars):.3f} BTC/bar "
              f"| toxic_rate mean {bars['toxic_rate'].mean():.3f}")

BTCUSDT week1: 10,071,946 ticks → 526,551 bars | avg 1.580 BTC/bar | toxic_rate mean 0.010
BTCUSDT week2: 12,032,260 ticks → 525,290 bars | avg 1.807 BTC/bar | toxic_rate mean 0.009
BTCUSDT week3: 24,249,845 ticks → 558,416 bars | avg 1.915 BTC/bar | toxic_rate mean 0.042
ETHUSDT week1: 4,554,411 ticks → 379,955 bars | avg 12.980 BTC/bar | toxic_rate mean 0.018
ETHUSDT week2: 7,137,925 ticks → 415,317 bars | avg 15.472 BTC/bar | toxic_rate mean 0.020
ETHUSDT week3: 10,051,489 ticks → 319,539 bars | avg 25.369 BTC/bar | toxic_rate mean 0.080
SOLUSDT week1: 3,968,322 ticks → 409,128 bars | avg 134.348 BTC/bar | toxic_rate mean 0.025
SOLUSDT week2: 5,757,031 ticks → 428,987 bars | avg 138.642 BTC/bar | toxic_rate mean 0.031
SOLUSDT week3: 11,434,844 ticks → 460,117 bars | avg 232.029 BTC/bar | toxic_rate mean 0.126


In [38]:
import numpy as np
def compute_gamma(bars: pd.DataFrame, target_q_critical_fills: float = 5.0) -> float:
    """
    Set gamma so quotes go uncompetitive only after target_q_critical_fills 
    average fills worth of inventory.
    
    q_critical = half_spread / (gamma * sigma * mid)
    gamma = half_spread / (q_critical * sigma * mid)
    """
    sigma = np.log(bars['mid']).diff().std()
    mid = bars['mid'].mean()
    spread = (bars['best_ask'] - bars['best_bid']).mean()
    half_spread = spread / 2
    
    # avg fill size per bar
    avg_fill = (bars['buy_volume'] / bars['buy_trades'].replace(0, np.nan)).mean()
    q_critical = avg_fill * target_q_critical_fills
    
    gamma = half_spread / (q_critical * sigma * mid)
    return gamma

# Compute per asset
for asset in ASSETS:
    bars = all_bars[(asset, 'week1')]
    g = compute_gamma(bars)
    print(f"{asset}: gamma={g:.6f}")

BTCUSDT: gamma=0.037720
ETHUSDT: gamma=0.006668
SOLUSDT: gamma=0.007950


In [39]:


# Run full table
results_all = {}
k_values = [0, 1, 2, 5, 10, 20, 50]
for asset in ASSETS:
    for week in ['week1']:
        bars = all_bars[(asset, week)]
        for k in k_values:
            res = run_backtest(
                bars=bars,
                k=k
            )
            stats = summary_stats(res)
            results_all[(asset, week)] = stats
            print(f"{asset} {week:5s} k : {k} | Sharpe {stats['sharpe']:8.2f} | "
                f"MtM {stats['final_mtm']:10.2f} | "
                f"inv_95 {stats['inventory_95pct']:.3f} | "
                f"fill_rate {stats['fill_rate']:.3f}")

BTCUSDT week1 k : 0 | Sharpe    -5.09 | MtM   -1937.31 | inv_95 2.024 | fill_rate 0.436
BTCUSDT week1 k : 1 | Sharpe    -5.31 | MtM   -2303.77 | inv_95 3.495 | fill_rate 0.409
BTCUSDT week1 k : 2 | Sharpe     1.67 | MtM    2100.00 | inv_95 3.488 | fill_rate 0.399
BTCUSDT week1 k : 5 | Sharpe     1.52 | MtM    2279.50 | inv_95 3.485 | fill_rate 0.398
BTCUSDT week1 k : 10 | Sharpe    -4.34 | MtM   -1482.08 | inv_95 2.020 | fill_rate 0.445
BTCUSDT week1 k : 20 | Sharpe    -5.78 | MtM   -2432.85 | inv_95 3.495 | fill_rate 0.414
BTCUSDT week1 k : 50 | Sharpe    -6.44 | MtM   -2800.75 | inv_95 3.495 | fill_rate 0.414
ETHUSDT week1 k : 0 | Sharpe   -35.34 | MtM    -405.80 | inv_95 2.156 | fill_rate 0.515
ETHUSDT week1 k : 1 | Sharpe    -6.52 | MtM    -237.36 | inv_95 2.324 | fill_rate 0.490
ETHUSDT week1 k : 2 | Sharpe    -7.20 | MtM    -244.57 | inv_95 2.324 | fill_rate 0.479
ETHUSDT week1 k : 5 | Sharpe    -4.70 | MtM    -199.36 | inv_95 2.593 | fill_rate 0.476
ETHUSDT week1 k : 10 | Sharpe

In [40]:
k_baseline = 0
k_adaptive = 2

print(f"{'Asset':<10} {'Week':<6} {'Baseline MtM':>14} {'Adaptive MtM':>14} {'Improvement':>12}")
print("-" * 60)

for asset in ASSETS:
    for week in WEEKS:
        bars = all_bars[(asset, week)]
        
        res_base = run_backtest(bars=bars, k=k_baseline)
        res_adap = run_backtest(bars=bars, k=k_adaptive)
        
        mtm_base = summary_stats(res_base)['final_mtm']
        mtm_adap = summary_stats(res_adap)['final_mtm']
        improvement = mtm_adap - mtm_base
        
        print(f"{asset:<10} {week:<6} {mtm_base:>14.2f} {mtm_adap:>14.2f} {improvement:>+12.2f}")

Asset      Week     Baseline MtM   Adaptive MtM  Improvement
------------------------------------------------------------
BTCUSDT    week1        -1937.31        2100.00     +4037.31
BTCUSDT    week2        -6109.53       -5567.01      +542.52
BTCUSDT    week3         -989.02         642.37     +1631.39
ETHUSDT    week1         -405.80        -244.57      +161.23
ETHUSDT    week2         -478.48        -548.49       -70.01
ETHUSDT    week3         -156.07         -92.07       +63.99
SOLUSDT    week1         -369.56        -326.00       +43.55
SOLUSDT    week2         -463.39        -405.42       +57.98
SOLUSDT    week3         -260.64         -56.68      +203.95
